# Task 1 and Task 3: Self-Sufficient Margin and TPC-DC Notebook

This notebook is self-contained and does not depend on `TypiClust-main/` at runtime.

The `Margin` and `TPC-DC` sections below implement the coursework methods directly in notebook form, following the paper's algorithmic structure and adapting the pipeline to run on Colab.

The Task 3 sections add a cached TPC-DC variant and compare it against the original baseline inside the same notebook.

## Colab Setup

Run in a GPU runtime.

In [ ]:
import sys
!{sys.executable} -m pip install -q torch torchvision scikit-learn matplotlib pandas tqdm faiss-cpu

In [ ]:
from __future__ import annotations

import copy
import random
import time
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Use a Colab GPU runtime."

device = torch.device("cuda")
OUTPUT_DIR = Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

## Settings

In [ ]:
BUDGETS = [20, 50, 100, 200, 500, 1000]
N_CLASSES = 10
MARGIN_SEED_SIZE = 20
MAX_CLUSTERS = 500
MIN_CLUSTER_SIZE = 5
K_NN = 20

SIMCLR_EPOCHS = 50
SCAN_EPOCHS = 10
CLASSIFIER_EPOCHS = 20

BATCH_SIZE = 128
NUM_WORKERS = 0
LEARNING_RATE = 1e-3
FEATURE_DIM = 128

## Data

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

simclr_view = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


class TwoViewDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, target = self.base_dataset[idx]
        return self.transform(image), self.transform(image), target, idx


class ImageOnlyDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, target = self.base_dataset[idx]
        return self.transform(image), target, idx


raw_train = datasets.CIFAR10(root="data", train=True, download=True)
raw_test = datasets.CIFAR10(root="data", train=False, download=True)
y_train = np.array(raw_train.targets)

simclr_dataset = TwoViewDataset(raw_train, simclr_view)
feature_dataset = ImageOnlyDataset(raw_train, eval_transform)
classifier_train_dataset = ImageOnlyDataset(raw_train, train_transform)
classifier_eval_dataset = ImageOnlyDataset(raw_train, eval_transform)
test_dataset = ImageOnlyDataset(raw_test, eval_transform)

## Shared Models

In [ ]:
def build_cifar_resnet18():
    backbone = models.resnet18(weights=None)
    backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    backbone.maxpool = nn.Identity()
    backbone.fc = nn.Identity()
    return backbone


class SimCLRModel(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM):
        super().__init__()
        self.encoder = build_cifar_resnet18()
        self.projector = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, feature_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, F.normalize(z, dim=1)


class ClusterModel(nn.Module):
    def __init__(self, encoder, num_clusters):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(512, num_clusters)

    def forward(self, x):
        h = self.encoder(x)
        return self.head(h), h


class ClassifierModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(512, N_CLASSES)

    def forward(self, x):
        return self.head(self.encoder(x))


def balanced_random_seed(labels, total_size, seed=SEED):
    rng = np.random.default_rng(seed)
    per_class = total_size // N_CLASSES
    extra = total_size % N_CLASSES
    selected = []
    for c in range(N_CLASSES):
        idx = np.flatnonzero(labels == c)
        take = per_class + (1 if c < extra else 0)
        selected.extend(rng.choice(idx, size=take, replace=False))
    return np.array(sorted(selected))


def select_random(budget):
    rng = np.random.default_rng(SEED + budget)
    return np.array(sorted(rng.choice(len(raw_train), size=budget, replace=False)))

## SimCLR Pretraining

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / temperature
    batch_size = z1.size(0)
    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)
    targets = torch.arange(batch_size, 2 * batch_size, device=z.device)
    targets = torch.cat([targets, torch.arange(batch_size, device=z.device)], dim=0)
    return F.cross_entropy(sim, targets)


def pretrain_simclr():
    model = SimCLRModel().to(device)
    loader = DataLoader(simclr_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(SIMCLR_EPOCHS):
        running = 0.0
        for x1, x2, _, _ in tqdm(loader, desc=f"SimCLR {epoch+1}/{SIMCLR_EPOCHS}", leave=False):
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss = nt_xent_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f"simclr epoch {epoch+1}: loss={running/len(loader):.4f}")
    return model


simclr_model = pretrain_simclr()

## Shared Evaluation Helpers

In [ ]:
def train_classifier(train_indices, epochs=CLASSIFIER_EPOCHS):
    model = ClassifierModel(copy.deepcopy(simclr_model.encoder)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(Subset(classifier_train_dataset, train_indices.tolist()), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    model.train()
    for epoch in range(epochs):
        running = 0.0
        for x, y, _ in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        if (epoch + 1) in {1, epochs}:
            print(f"classifier epoch {epoch+1}: loss={running/len(train_loader):.4f}")
    return model


@torch.no_grad()
def evaluate_classifier(model):
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    correct = 0
    total = 0
    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

## Margin Section

This section implements margin sampling in notebook form using the same acquisition idea described in the coursework materials.

In [ ]:
@torch.no_grad()
def margin_query(budgetSize, lSet, uSet, model):
    clf = model.to(device)
    u_ranks = []
    uSetLoader = DataLoader(Subset(classifier_eval_dataset, uSet.tolist()), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    print("len(uSetLoader): {}".format(len(uSetLoader)))
    for i, (x_u, _, idx) in enumerate(tqdm(uSetLoader, desc="uSet Activations", leave=False)):
        x_u = x_u.to(device, non_blocking=True)
        temp_u_rank = torch.softmax(clf(x_u), dim=1)
        temp_u_rank, _ = torch.sort(temp_u_rank, descending=True)
        difference = temp_u_rank[:, 0] - temp_u_rank[:, 1]
        difference = -1 * difference
        u_ranks.append(difference.detach().cpu().numpy())
    u_ranks = np.concatenate(u_ranks, axis=0)
    print(f"u_ranks.shape: {u_ranks.shape}")
    sorted_idx = np.argsort(u_ranks)[::-1]
    activeSet = uSet[sorted_idx[:budgetSize]]
    remainSet = uSet[sorted_idx[budgetSize:]]
    return activeSet, remainSet


def select_margin(budget):
    if budget <= MARGIN_SEED_SIZE:
        return balanced_random_seed(y_train, budget)
    lSet = balanced_random_seed(y_train, MARGIN_SEED_SIZE)
    uSet = np.setdiff1d(np.arange(len(raw_train)), lSet)
    seed_model = train_classifier(lSet, epochs=max(8, CLASSIFIER_EPOCHS // 2))
    activeSet, _ = margin_query(budget - MARGIN_SEED_SIZE, lSet, uSet, seed_model)
    return np.array(sorted(np.concatenate([lSet, activeSet])))


def run_margin_only(budget):
    random_sel = select_random(budget)
    random_acc = evaluate_classifier(train_classifier(random_sel))
    margin_sel = select_margin(budget)
    margin_acc = evaluate_classifier(train_classifier(margin_sel))
    return {"budget": budget, "random_acc": random_acc, "margin_acc": margin_acc, "diff_pct": 100.0 * (margin_acc - random_acc)}

## TPC-DC Section

This section implements TPC-DC in notebook form using SimCLR features, SCAN-style clustering, and density-based typicality inside each cluster.

In [ ]:
@torch.no_grad()
def extract_features(model, dataset, batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    feats = []
    indices = []
    model.eval()
    for x, _, idx in tqdm(loader, desc="Extract features", leave=False):
        x = x.to(device, non_blocking=True)
        h = model.encoder(x)
        feats.append(h.cpu())
        indices.append(idx)
    return torch.cat(feats).numpy(), torch.cat(indices).numpy()


base_features, feature_indices = extract_features(simclr_model, feature_dataset)
assert np.array_equal(feature_indices, np.arange(len(feature_dataset)))


class NeighborPairDataset(Dataset):
    def __init__(self, base_dataset, neighbors):
        self.base_dataset = base_dataset
        self.neighbors = neighbors

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _, _ = self.base_dataset[idx]
        nn_idx = int(np.random.choice(self.neighbors[idx]))
        nn_img, _, _ = self.base_dataset[nn_idx]
        return img, nn_img


def scan_loss(logits_a, logits_b, entropy_weight=2.0):
    p_a = torch.softmax(logits_a, dim=1)
    p_b = torch.softmax(logits_b, dim=1)
    agreement = torch.sum(p_a * p_b, dim=1)
    consistency = -torch.log(agreement + 1e-8).mean()
    p_mean = p_a.mean(dim=0)
    entropy = -torch.sum(p_mean * torch.log(p_mean + 1e-8))
    return consistency - entropy_weight * entropy


def train_scan_clusterer(num_clusters):
    nbrs = NearestNeighbors(n_neighbors=K_NN + 1, metric="euclidean")
    nbrs.fit(base_features)
    _, neighbor_idx = nbrs.kneighbors(base_features)
    neighbor_idx = neighbor_idx[:, 1:]

    pair_dataset = NeighborPairDataset(feature_dataset, neighbor_idx)
    pair_loader = DataLoader(pair_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    cluster_model = ClusterModel(copy.deepcopy(simclr_model.encoder), num_clusters).to(device)
    optimizer = torch.optim.Adam(cluster_model.parameters(), lr=LEARNING_RATE)
    cluster_model.train()
    for epoch in range(SCAN_EPOCHS):
        running = 0.0
        for xa, xb in tqdm(pair_loader, desc=f"SCAN {epoch+1}/{SCAN_EPOCHS}", leave=False):
            xa = xa.to(device, non_blocking=True)
            xb = xb.to(device, non_blocking=True)
            logits_a, _ = cluster_model(xa)
            logits_b, _ = cluster_model(xb)
            loss = scan_loss(logits_a, logits_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f"scan epoch {epoch+1}: loss={running/len(pair_loader):.4f}")
    return cluster_model


def get_nn(features, num_neighbors):
    d = features.shape[1]
    features = features.astype(np.float32)
    index = faiss.IndexFlatL2(d)
    index.add(features)
    distances, indices = index.search(features, num_neighbors + 1)
    return distances[:, 1:], indices[:, 1:]


def get_mean_nn_dist(features, num_neighbors, return_indices=False):
    distances, indices = get_nn(features, num_neighbors)
    mean_distance = distances.mean(axis=1)
    if return_indices:
        return mean_distance, indices
    return mean_distance


def calculate_typicality(features, num_neighbors):
    mean_distance = get_mean_nn_dist(features, num_neighbors)
    typicality = 1 / (mean_distance + 1e-5)
    return typicality


@torch.no_grad()
def infer_cluster_labels(cluster_model):
    loader = DataLoader(feature_dataset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    logits_list = []
    for x, _, _ in tqdm(loader, desc="Infer clusters", leave=False):
        x = x.to(device, non_blocking=True)
        logits, _ = cluster_model(x)
        logits_list.append(logits.cpu())
    logits = torch.cat(logits_list).numpy()
    return logits.argmax(axis=1)


def select_tpc_dc(budget):
    num_clusters = min(budget, MAX_CLUSTERS)
    cluster_model = train_scan_clusterer(num_clusters)
    clusters = infer_cluster_labels(cluster_model)

    lSet = np.array([], dtype=int)
    uSet = np.arange(len(raw_train), dtype=int)
    relevant_indices = np.concatenate([lSet, uSet]).astype(int)
    features = base_features[relevant_indices]
    labels = np.copy(clusters[relevant_indices])
    existing_indices = np.arange(len(lSet))

    cluster_ids, cluster_sizes = np.unique(labels, return_counts=True)
    cluster_labeled_counts = np.bincount(labels[existing_indices], minlength=len(cluster_ids)) if len(existing_indices) else np.zeros(len(cluster_ids), dtype=int)
    clusters_df = pd.DataFrame({
        "cluster_id": cluster_ids,
        "cluster_size": cluster_sizes,
        "existing_count": cluster_labeled_counts,
        "neg_cluster_size": -1 * cluster_sizes,
    })
    clusters_df = clusters_df[clusters_df.cluster_size > MIN_CLUSTER_SIZE]
    clusters_df = clusters_df.sort_values(["existing_count", "neg_cluster_size"])
    labels[existing_indices] = -1

    selected = []
    for i in range(budget):
        cluster = clusters_df.iloc[i % len(clusters_df)].cluster_id
        indices = (labels == cluster).nonzero()[0]
        rel_feats = features[indices]
        typicality = calculate_typicality(rel_feats, min(K_NN, max(1, len(indices) // 2)))
        idx = indices[typicality.argmax()]
        selected.append(idx)
        labels[idx] = -1

    selected = np.array(selected)
    activeSet = relevant_indices[selected]
    return activeSet


def run_tpcdc_only(budget):
    random_sel = select_random(budget)
    random_acc = evaluate_classifier(train_classifier(random_sel))
    tpc_sel = select_tpc_dc(budget)
    tpc_acc = evaluate_classifier(train_classifier(tpc_sel))
    return {"budget": budget, "random_acc": random_acc, "tpc_acc": tpc_acc, "diff_pct": 100.0 * (tpc_acc - random_acc)}

## Task 3 Modification: Cached TPC-DC

This section leaves the original TPC-DC baseline unchanged and adds a refresh-cache variant that reuses per-cluster rankings for a limited number of revisits before recomputing them.

In [ ]:
def build_clusters_df(labels, existing_indices):
    cluster_ids, cluster_sizes = np.unique(labels, return_counts=True)
    cluster_labeled_counts = (
        np.bincount(labels[existing_indices], minlength=len(cluster_ids))
        if len(existing_indices)
        else np.zeros(len(cluster_ids), dtype=int)
    )
    clusters_df = pd.DataFrame({
        "cluster_id": cluster_ids,
        "cluster_size": cluster_sizes,
        "existing_count": cluster_labeled_counts,
        "neg_cluster_size": -1 * cluster_sizes,
    })
    clusters_df = clusters_df[clusters_df.cluster_size > MIN_CLUSTER_SIZE]
    clusters_df = clusters_df.sort_values(["existing_count", "neg_cluster_size"])
    return clusters_df


def build_cluster_rank_cache(labels, features):
    cluster_cache = {}
    for cluster in np.unique(labels):
        if cluster < 0:
            continue
        indices = (labels == cluster).nonzero()[0]
        if len(indices) == 0:
            continue
        rel_feats = features[indices]
        typicality = calculate_typicality(rel_feats, min(K_NN, max(1, len(indices) // 2)))
        ranked = indices[np.argsort(typicality)[::-1]]
        cluster_cache[int(cluster)] = ranked
    return cluster_cache


def refresh_cluster_ranking(cluster, labels, features, cluster_cache, refresh_counts):
    indices = (labels == cluster).nonzero()[0]
    if len(indices) == 0:
        cluster_cache[int(cluster)] = np.array([], dtype=int)
        refresh_counts[int(cluster)] = 0
        return

    rel_feats = features[indices]
    typicality = calculate_typicality(rel_feats, min(K_NN, max(1, len(indices) // 2)))
    ranked = indices[np.argsort(typicality)[::-1]]
    cluster_cache[int(cluster)] = ranked
    refresh_counts[int(cluster)] = 0


def select_tpc_dc_from_clusters(budget, features, clusters):
    lSet = np.array([], dtype=int)
    uSet = np.arange(len(raw_train), dtype=int)
    relevant_indices = np.concatenate([lSet, uSet]).astype(int)
    features_local = features[relevant_indices]
    labels = np.copy(clusters[relevant_indices])
    existing_indices = np.arange(len(lSet))

    clusters_df = build_clusters_df(labels, existing_indices)
    labels[existing_indices] = -1

    selected = []
    for i in range(budget):
        cluster = clusters_df.iloc[i % len(clusters_df)].cluster_id
        indices = (labels == cluster).nonzero()[0]
        rel_feats = features_local[indices]
        typicality = calculate_typicality(rel_feats, min(K_NN, max(1, len(indices) // 2)))
        idx = indices[typicality.argmax()]
        selected.append(idx)
        labels[idx] = -1

    return relevant_indices[np.array(selected)]


def select_tpc_dc_cached_refresh_from_clusters(
    budget,
    features,
    clusters,
    refresh_every=8,
    refresh_fraction=0.10,
):
    lSet = np.array([], dtype=int)
    uSet = np.arange(len(raw_train), dtype=int)
    relevant_indices = np.concatenate([lSet, uSet]).astype(int)
    features_local = features[relevant_indices]
    labels = np.copy(clusters[relevant_indices])
    existing_indices = np.arange(len(lSet))

    clusters_df = build_clusters_df(labels, existing_indices)
    labels[existing_indices] = -1

    cluster_cache = build_cluster_rank_cache(labels, features_local)
    cluster_offsets = {int(cluster_id): 0 for cluster_id in clusters_df.cluster_id.tolist()}
    refresh_counts = {int(cluster_id): 0 for cluster_id in clusters_df.cluster_id.tolist()}
    initial_cluster_sizes = {
        int(cluster_id): int((labels == cluster_id).sum())
        for cluster_id in clusters_df.cluster_id.tolist()
    }

    selected = []
    for i in range(budget):
        cluster = int(clusters_df.iloc[i % len(clusters_df)].cluster_id)

        current_size = int((labels == cluster).sum())
        if current_size == 0:
            continue

        removed = initial_cluster_sizes[cluster] - current_size
        needs_fraction_refresh = removed >= max(1, int(refresh_fraction * initial_cluster_sizes[cluster]))
        needs_periodic_refresh = refresh_counts[cluster] >= refresh_every
        cache_empty = cluster not in cluster_cache or len(cluster_cache[cluster]) == 0

        if cache_empty or needs_periodic_refresh or needs_fraction_refresh:
            refresh_cluster_ranking(cluster, labels, features_local, cluster_cache, refresh_counts)
            cluster_offsets[cluster] = 0
            initial_cluster_sizes[cluster] = max(initial_cluster_sizes[cluster], current_size)

        ranked = cluster_cache[cluster]
        ptr = cluster_offsets[cluster]
        while ptr < len(ranked) and labels[ranked[ptr]] == -1:
            ptr += 1
        if ptr >= len(ranked):
            refresh_cluster_ranking(cluster, labels, features_local, cluster_cache, refresh_counts)
            cluster_offsets[cluster] = 0
            ranked = cluster_cache[cluster]
            ptr = 0
            while ptr < len(ranked) and labels[ranked[ptr]] == -1:
                ptr += 1
            if ptr >= len(ranked):
                continue

        idx = ranked[ptr]
        selected.append(idx)
        labels[idx] = -1
        cluster_offsets[cluster] = ptr + 1
        refresh_counts[cluster] += 1

    selected = np.array(selected)
    if len(selected) < budget:
        remaining = np.setdiff1d(np.arange(len(relevant_indices)), selected, assume_unique=False)
        selected = np.concatenate([selected, remaining[: budget - len(selected)]])
    return relevant_indices[selected]

## Task 3 Evaluation

This section compares the original TPC-DC selection routine with the refresh-cache variant while timing only the selection step on fixed cluster assignments.

In [ ]:
TASK3_BUDGETS = [200, 500, 1000]
task3_rows = []
for budget in TASK3_BUDGETS:
    print(f"\n=== Budget {budget} ===")
    num_clusters = min(budget, MAX_CLUSTERS)
    cluster_model = train_scan_clusterer(num_clusters)
    clusters = infer_cluster_labels(cluster_model)

    t0 = time.perf_counter()
    original_sel = select_tpc_dc_from_clusters(budget, base_features, clusters)
    original_select_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    cached_sel = select_tpc_dc_cached_refresh_from_clusters(
        budget,
        base_features,
        clusters,
        refresh_every=8,
        refresh_fraction=0.10,
    )
    cached_select_time = time.perf_counter() - t0

    task3_rows.append({
        "budget": budget,
        "same_selection": np.array_equal(np.sort(original_sel), np.sort(cached_sel)),
        "original_select_time_sec": original_select_time,
        "cached_select_time_sec": cached_select_time,
        "time_speedup": original_select_time / cached_select_time,
    })

task3_results = pd.DataFrame(task3_rows)
task3_results.to_csv(OUTPUT_DIR / "task3_selection_only_timing.csv", index=False)
task3_results

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=150)

ax.plot(task3_results["budget"], task3_results["original_select_time_sec"], marker="o", linewidth=3, color="tab:purple", label="Original selection")
ax.plot(task3_results["budget"], task3_results["cached_select_time_sec"], marker="o", linewidth=3, color="tab:green", label="Refresh-cache selection")
ax.set_xscale("symlog", linthresh=25)
ax.set_xlabel("Number of examples")
ax.set_ylabel("Selection time (seconds)")
ax.set_title("Task 3 Selection-Only Runtime")
ax.grid(True, which="both", alpha=0.2)
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "task3_selection_only_timing.png", bbox_inches="tight")
task3_results

## Combined Comparison

In [ ]:
rows = []
for budget in BUDGETS:
    random_sel = select_random(budget)
    random_acc = evaluate_classifier(train_classifier(random_sel))
    margin_sel = select_margin(budget)
    margin_acc = evaluate_classifier(train_classifier(margin_sel))
    tpc_sel = select_tpc_dc(budget)
    tpc_acc = evaluate_classifier(train_classifier(tpc_sel))

    rows.append({"method": "No AL", "num_examples": budget, "accuracy": random_acc})
    rows.append({"method": "Margin", "num_examples": budget, "accuracy": margin_acc})
    rows.append({"method": "TPC_DC", "num_examples": budget, "accuracy": tpc_acc})

results = pd.DataFrame(rows)
results.to_csv(OUTPUT_DIR / "task1_results.csv", index=False)
results

## Plot

In [ ]:
baseline = results[results.method == "No AL"][["num_examples", "accuracy"]].rename(columns={"accuracy": "baseline_accuracy"})
plot_df = results.merge(baseline, on="num_examples")
plot_df["accuracy_difference"] = 100.0 * (plot_df["accuracy"] - plot_df["baseline_accuracy"])

colors = {"No AL": "black", "Margin": "tab:orange", "TPC_DC": "tab:purple"}
labels = {"No AL": "No AL", "Margin": "Margin", "TPC_DC": r"TPC$_{DC}$"}

fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=150)
for method in ["No AL", "Margin", "TPC_DC"]:
    sub = plot_df[plot_df.method == method].sort_values("num_examples")
    ax.plot(sub["num_examples"], sub["accuracy_difference"], marker="o", linewidth=3, markersize=6, color=colors[method], label=labels[method])
ax.axhline(0, color="black", linewidth=1, alpha=0.35)
ax.set_xscale("symlog", linthresh=25)
ax.set_xlabel("Number of examples", fontsize=14)
ax.set_ylabel("Accuracy difference", fontsize=14)
ax.set_title("CIFAR-10: implemented Margin and TPC-DC", fontsize=13)
ax.grid(True, which="both", alpha=0.2)
ax.legend(frameon=False, ncol=2, loc="lower left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "task1_plot.png", bbox_inches="tight")
plot_df

In [ ]:
plot_df.to_csv("task1_results.csv", index=False)